In [17]:
import torch
import torch.nn as nn
import nbimporter
import numpy as np
from FNN import FeedForwardNeuralNetwork as FNN

In [18]:
print(torch.__version__)

2.11.0+cpu


## Transform

In [19]:
class MyTransformer(nn.Module):
    def __init__(self, input_dim, output_dim, heads_num, num_layers, d_model):
        super(MyTransformer, self).__init__()
        # embedding
        self.embedding = nn.Embedding(input_dim, d_model)
        # 暂时假设最大文本长度为10000
        self.p_encoding = PositionalEncoding(d_model, 10000)
        self.encoder = Encoder(input_dim, heads_num, num_layers, d_model)

    def forward(self, x):
        x = self.embedding(x)
        x = self.p_encoding(x)
        return x
    

## 编码器(Encoder)

In [20]:
class Encoder(nn.Module):
    def __init__(self, input_dim, head_num, head_dim, d_model):
        super(Encoder, self).__init__()
        self.multhattention = MultiHeadAttention(head_num, head_dim, d_model)
        self.fnn = FNN()
        self.norm_multhead = nn.LayerNorm(d_model)
        self.norm_fnn = nn.LayerNorm(d_model)

    def forward(self, x):
        temp_x = x
        x = multhattention(x)
        x = x + temp_x
        x = x.norm_multhead(x)

        temp_x = x
        x = fnn(x)
        x = x + temp_x
        x = x.norm_fnn(x)
        return x

In [57]:
# test
x = torch.randn(2, 10, 512)
encoder = Encoder(512, 8, 64, 512)
x = encoder(x)
print(x.shape)

torch.Size([2, 10, 512])


## 自注意力模块

In [50]:
class SelfAttention(nn.Module):
    def __init__(self, d_k, d_v, d_model):
        # d_k一般等于d_v
        super(SelfAttention, self).__init__()
        # Q,K,V参数矩阵
        self.w_query = nn.Linear(d_model, d_k)
        self.w_key = nn.Linear(d_model, d_k)
        self.w_value = nn.Linear(d_model, d_v)
        re_sqr_d_k = torch.exp(torch.tensor(-0.5 * np.log(d_k), dtype=torch.float32))
        # 注册为缓冲，方便并行计算
        self.register_buffer("re_sqr_d_k", re_sqr_d_k)

    def forward(self, x):
        # x的结构为[ batch_size, seq_len, d_model ]
        q = self.w_query(x)
        k = self.w_key(x)
        v = self.w_value(x)
        x = self.attention(q,k,v)
        return x

    def attention(self, q, k, v):
        return torch.matmul(torch.softmax( torch.matmul(q, k.transpose (-2,-1)) * self.re_sqr_d_k , dim=-1), v)

### 多头注意力

In [55]:
class MultiHeadAttention(nn.Module):
    def __init__(self, head_num, head_dim, d_model):
        # head_dim = d_k = d_v
        # d_model = head_num * d_k
        super(MultiHeadAttention, self).__init__()
        self.head_num = head_num
        self.head_dim = head_dim
        self.w_query = nn.Linear(d_model, head_num * head_dim)
        self.w_key = nn.Linear(d_model, head_num * head_dim)
        self.w_value = nn.Linear(d_model, head_num * head_dim)
        # W_O用来将多头结果整合
        self.w_output = nn.Linear(head_num * head_dim, d_model)
        re_sqr_d_k_multi = torch.exp(torch.tensor(-0.5 * np.log(head_dim), dtype=torch.float32))
        # 注册为缓冲，方便并行计算
        self.register_buffer("re_sqr_d_k_multi", re_sqr_d_k_multi)

    def forward(self, x):
        # x的结构为[ batch_size, seq_len, d_model ]
        # x先和q,k,v矩阵做乘法
        # 再将qkv切成head_num个头
        q = self.w_query(x)
        k = self.w_key(x)
        v = self.w_value(x)
        batch_size = x.shape[0]
        # qkv矩阵的结构为[batch_size, seq_len, head_num * head_dim]
        # 只拆最后一维
        q = q.reshape(batch_size, -1, self.head_num, self.head_dim)
        # 交换第2维和第3维
        q = q.transpose(1,2)
        
        k = k.reshape(batch_size, -1, self.head_num, self.head_dim)
        # 交换第2维和第3维
        k = k.transpose(1,2)
        
        v = v.reshape(batch_size, -1, self.head_num, self.head_dim)
        # 交换第2维和第3维
        v = v.transpose(1,2)

        x = self.attention(q,k,v)

        x = x.transpose(1,2)
        # 重新连接成d_model大小
        x = x.reshape(batch_size, -1, self.head_num * self.head_dim)
        x = self.w_output(x)
        
        print(x.shape)
        return x

    def attention(self, q, k, v):
        return torch.matmul(torch.softmax( torch.matmul(q, k.transpose (-2,-1)) * self.re_sqr_d_k_multi , dim=-1), v)

In [56]:
# test
ma = MultiHeadAttention(8, 64, 512)
print(ma.w_query)
print(ma.w_key)
print(ma.w_value)
x = torch.randn(2, 10, 512)
ma(x)

Linear(in_features=512, out_features=512, bias=True)
Linear(in_features=512, out_features=512, bias=True)
Linear(in_features=512, out_features=512, bias=True)
torch.Size([2, 10, 512])


tensor([[[-1.6270e-01,  4.6352e-02,  1.6336e-01,  ..., -4.3353e-02,
           1.5266e-02,  6.4731e-02],
         [-1.8200e-01,  4.1711e-02,  1.6132e-01,  ..., -2.8505e-03,
           5.7191e-02,  3.4110e-02],
         [-1.8394e-01,  3.4819e-02,  1.7405e-01,  ..., -5.6109e-02,
           7.2670e-02, -2.0611e-02],
         ...,
         [-2.1804e-01,  9.2461e-03,  1.9206e-01,  ..., -7.9840e-02,
           1.0979e-01, -4.5367e-02],
         [-1.6257e-01,  6.4422e-02,  1.6566e-01,  ..., -1.6815e-01,
           1.1735e-04,  5.7264e-02],
         [-1.6088e-01,  7.8842e-02,  1.7731e-01,  ..., -7.0018e-02,
           5.6182e-02,  2.5054e-02]],

        [[-1.9039e-01, -1.1641e-01, -5.8658e-02,  ...,  5.3452e-02,
           2.9342e-02, -1.3675e-02],
         [-1.5482e-01, -8.2485e-02, -4.4295e-02,  ...,  1.3211e-01,
           7.7985e-02, -9.7689e-03],
         [-1.4334e-01, -1.2225e-01, -4.1685e-02,  ...,  1.1003e-01,
           1.6327e-02, -1.5016e-02],
         ...,
         [-1.5649e-01, -7

## 位置编码器

In [37]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()
        # 空的位置编码空间
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, 1, dtype=torch.float).unsqueeze(1) #在1维加上一维，使数据成为二位数组，方便进行矩阵相乘
        denom = torch.exp((-np.log(10000) / d_model) * torch.arange(0, d_model, 2).float())  # 2i和2i+1位置都要乘以2i，因此步长为2

        # 计算位置编码
        pe[:, 0::2] = torch.sin(position * denom) #偶数位置用sin
        pe[:, 1::2] = torch.cos(position * denom) #奇数位置用cos

        pe = pe.unsqueeze(0)

        # 模型中位置编码是固定的，因此可以注册为定值
        self.register_buffer('pe', pe)
    def forward(self, x):
        # 直接将pe与embedding好的数据相加即可
        return x + self.pe[:, :x.size(1), :]

In [38]:
model = MyTransformer(input_dim=1000, output_dim=10, heads_num=8, num_layers=6, d_model=512)
test_input = torch.randint(0, 1000, (2, 10))  # [batch=2, seq_len=10]
output = model(test_input)
print(output.shape)  # 输出: torch.Size([2, 10, 512])

torch.Size([2, 10, 512])
